# Stage 4: Pandas Analysis – TMDB Movie Dataset

## Project Overview
This notebook focuses on performing Exploratory Data Analysis (EDA) using Python Pandas on the cleaned TMDB movie datasets.

The objective is to extract meaningful insights, identify trends, analyze movie performance, and answer business-related questions using Pandas operations.

## Objectives
- Load the cleaned datasets using Pandas.
- Perform data exploration and statistical analysis.
- Analyze movie budgets, revenues, ratings, genres, cast, crew, and keywords.
- Identify patterns and trends in the movie industry.
- Answer the 10 project questions using Pandas.

## Tools and Libraries
- Python 3.12.0
- Pandas
- NumPy
- Matplotlib
- Seaborn

## Dataset Source
The cleaned datasets from Stage 1 are stored in the `cleaned_data/` directory.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
movies_df = pd.read_csv("cleaned_data/movies.csv")
genres_df = pd.read_csv("cleaned_data/genres.csv")
movie_genres_df = pd.read_csv("cleaned_data/movie_genres.csv")
cast_df = pd.read_csv("cleaned_data/cast.csv")
crew_df = pd.read_csv("cleaned_data/crew.csv")
movie_keywords_df = pd.read_csv("cleaned_data/movie_keywords.csv")

In [3]:
datasets = {
    "Movies": movies_df,
    "Genres": genres_df,
    "Movie Genres": movie_genres_df,
    "Cast": cast_df,
    "Crew": crew_df,
    "Movie Keywords": movie_keywords_df
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Movies: (2503, 12)
Genres: (19, 2)
Movie Genres: (6878, 2)
Cast: (24902, 6)
Crew: (7289, 6)
Movie Keywords: (41681, 3)


## Dataset Loading and Verification

### Objective
Load all six cleaned TMDB datasets into Pandas DataFrames and verify that they are successfully imported.

### Approach
- Used `pd.read_csv()` to load the cleaned CSV files from the `cleaned_data/` directory.
- Stored each dataset in a separate Pandas DataFrame.
- Used the `.shape` attribute to verify the number of rows and columns in each dataset.

### Observation
All six cleaned datasets were successfully loaded into their respective DataFrames. Their shapes were verified against the expected row counts from the data-cleaning stage.

### Conclusion
The cleaned TMDB datasets are successfully loaded and ready for further analysis using Pandas.

### Problem 1: Count Unique Records Across All Tables

**Objective:**

Calculate the total number of unique movies, genres, cast members, and crew members available in the cleaned TMDB datasets.

In [4]:
# Count unique records in each dataset

unique_movies = movies_df["movie_id"].nunique()
unique_genres = genres_df["genre_id"].nunique()
unique_cast_members = cast_df["person_id"].nunique()
unique_crew_members = crew_df["person_id"].nunique()

# Display the results

print("Total Unique Movies:", unique_movies)
print("Total Unique Genres:", unique_genres)
print("Total Unique Cast Members:", unique_cast_members)
print("Total Unique Crew Members:", unique_crew_members)

Total Unique Movies: 2503
Total Unique Genres: 19
Total Unique Cast Members: 11300
Total Unique Crew Members: 2848


**Approach:**
- Used the `nunique()` method to count distinct IDs in each dataset.
- Used `movie_id` to identify unique movies.
- Used `genre_id` to identify unique genres.
- Used `person_id` to identify unique cast and crew members.

**Observation:**

| Category | Unique Count |
|---|---:|
| Movies | 2,503 |
| Genres | 19 |
| Cast Members | 11,300 |
| Crew Members | 2,848 |

The movies dataset contains 2,503 unique movies and 19 unique genres. The cast and crew datasets contain 11,300 and 2,848 unique individuals, respectively.

**Conclusion:**

The unique record counts provide an overview of the size and diversity of the TMDB dataset across movies, genres, cast members, and crew members.

### Problem 2 - List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis

In [5]:
# Load the original movies dataset
original_movies_df = pd.read_csv("data/movies.csv")

# Find movies with missing or zero budget/revenue
problematic_movies = original_movies_df[
    (original_movies_df["budget"].isna()) |
    (original_movies_df["budget"] == 0) |
    (original_movies_df["revenue"].isna()) |
    (original_movies_df["revenue"] == 0)
]

# Display the affected movies
problematic_movies[
    ["movie_id", "title", "budget", "revenue"]
]

,movie_id,title,budget,revenue
324,791373,Zack Snyder's Justice League,70000000.0,0.0
340,405774,Bird Box,19800000.0,0.0
403,664413,365 Days,0.0,9458590.0
468,466282,To All the Boys I've Loved Before,0.0,0.0
554,766507,Prey,65000000.0,0.0
...,...,...,...,...
2469,755566,Day Shift,100000000.0,0.0
2471,73861,A Serbian Film,0.0,0.0
2480,11104,Chungking Express,160000.0,0.0
2481,500840,I'm Thinking of Ending Things,0.0,0.0


In [6]:
# Count missing and zero values

print("Missing Budget:", original_movies_df["budget"].isna().sum())
print("Zero Budget:", (original_movies_df["budget"] == 0).sum())

print("Missing Revenue:", original_movies_df["revenue"].isna().sum())
print("Zero Revenue:", (original_movies_df["revenue"] == 0).sum())

print("Total Affected Movies:", len(problematic_movies))

Missing Budget: 0
Zero Budget: 90
Missing Revenue: 0
Zero Revenue: 102
Total Affected Movies: 133




**Approach:**
- Loaded the original movies dataset to inspect the values before cleaning.
- Used `isna()` to identify missing values.
- Used conditional filtering (`== 0`) to identify zero budget and revenue values.
- Used `sum()` to count the affected records.
- Used `to_string()` to display all affected movie records without truncation.

**Observation:**

| Category | Count |
|---|---:|
| Missing Budget | 0 |
| Zero Budget | 90 |
| Missing Revenue | 0 |
| Zero Revenue | 102 |
| Total Affected Movies | 133 |
| Movies with Both Budget and Revenue Zero | 59 |

The dataset contains 90 movies with zero budget and 102 movies with zero revenue. A total of 133 unique movies are affected because 59 movies have both values recorded as zero.

**Data Treatment:**

- Missing and zero budget/revenue values are treated as unavailable financial information.
- Zero values were converted to `NaN` during the data-cleaning stage.
- Missing values were subsequently replaced with the respective column medians.
- Original valid financial values were retained.

**Conclusion:**

The original dataset contained 133 movies with incomplete financial information. Median imputation was applied during data cleaning to maintain a complete dataset for further analysis. However, financial calculations such as profit and ROI must be interpreted cautiously because some values are estimated rather than actual reported figures.

### Problem 3 – List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.

In [8]:
# Select movies with valid, positive budget and revenue

valid_movies = original_movies_df[
    (original_movies_df["budget"].notna()) &
    (original_movies_df["revenue"].notna()) &
    (original_movies_df["budget"] > 0) &
    (original_movies_df["revenue"] > 0)
].copy()

# Calculate profit
valid_movies["profit"] = (
    valid_movies["revenue"] - valid_movies["budget"]
)

# Calculate ROI as a percentage
valid_movies["ROI"] = (
    valid_movies["profit"] / valid_movies["budget"]
) * 100

# Display the results
valid_movies[
    ["movie_id", "title", "budget", "revenue", "profit", "ROI"]
]

,movie_id,title,budget,revenue,profit,ROI
0,157336,Interstellar,1.650000e+08,7.466067e+08,5.816067e+08,352.488913
1,27205,Inception,1.600000e+08,8.390306e+08,6.790306e+08,424.394144
2,24428,The Avengers,2.200000e+08,1.518816e+09,1.298816e+09,590.370689
3,155,The Dark Knight,1.850000e+08,1.004558e+09,8.195584e+08,443.004564
4,19995,Avatar,2.370000e+08,2.923706e+09,2.686706e+09,1133.631235
...,...,...,...,...,...,...
2504,98,Gladiator,1.030000e+08,4.655162e+08,3.625162e+08,351.957522
2505,1368167,Date Night,9.321480e+07,8.759568e+07,-5.619119e+06,-6.028140
2506,1368168,Transformers,1.100988e+08,2.747467e+08,1.646479e+08,149.545660
2507,1368169,The SpongeBob Movie: Sponge Out of Water,3.928488e+07,5.706575e+08,5.313726e+08,1352.613653


**Approach:**
- Filtered movies with budget and revenue greater than zero.
- Used `.copy()` to create an independent DataFrame for calculations.
- Calculated profit by subtracting budget from revenue.
- Calculated ROI as a percentage using profit divided by budget, multiplied by 100.

**Formulas:**

Profit = Revenue − Budget

ROI = (Profit / Budget) × 100

**Observation:**
- A total of 2,376 movies had valid, positive budget and revenue values.
- Avatar recorded an ROI of approximately 1,133.63%.
- The Avengers recorded an ROI of approximately 590.37%.
- Date Night recorded a negative ROI of approximately -6.03%, indicating that its recorded revenue was lower than its production budget.
- Both positive and negative ROI values were observed.

**Conclusion:**

The analysis helps evaluate the financial performance of movies by comparing their revenue against production budgets. However, this ROI represents a simplified measure and does not account for marketing, distribution, or other expenses.

### Problem 4: Identify the Top 15 Movies by Return on Investment

**Objective:**

Identify the 15 movies with the highest Return on Investment (ROI), considering only movies with a production budget of at least $1 million.

**Approach:**
- Filter movies with a budget of at least $1,000,000.
- Exclude movies with missing or zero revenue.
- Calculate profit and ROI.
- Sort movies by ROI in descending order.
- Select the top 15 movies.

In [9]:
# Filter movies with budget >= $1 million
top_roi_movies = original_movies_df[
    (original_movies_df["budget"] >= 1000000) &
    (original_movies_df["revenue"] > 0)
].copy()

# Calculate profit
top_roi_movies["profit"] = (
    top_roi_movies["revenue"] - top_roi_movies["budget"]
)

# Calculate ROI
top_roi_movies["ROI"] = (
    top_roi_movies["profit"] / top_roi_movies["budget"]
) * 100

# Sort by ROI and select top 15 movies
top_15_roi = top_roi_movies.sort_values(
    by="ROI",
    ascending=False
).head(15)

# Display results
top_15_roi[
    ["movie_id", "title", "budget", "revenue", "profit", "ROI"]
]

,movie_id,title,budget,revenue,profit,ROI
1767,503314,Dragon Ball Super: Broly,1000000.0,125002821.0,124002821.0,12400.282100
535,408,Snow White and the Seven Dwarfs,1488423.0,184925486.0,183437063.0,12324.256142
1791,36685,The Rocky Horror Picture Show,1400000.0,171181400.0,169781400.0,12127.242857
467,1366,Rocky,1000000.0,117253345.0,116253345.0,11625.334500
1227,770,Gone with the Wind,4000000.0,402352579.0,398352579.0,9958.814475
716,9325,The Jungle Book,4000000.0,378000000.0,374000000.0,9350.000000
635,11224,Cinderella,2900000.0,263600000.0,260700000.0,8989.655172
372,176,Saw,1200000.0,104045735.0,102845735.0,8570.477917
719,12230,One Hundred and One Dalmatians,3600000.0,303000000.0,299400000.0,8316.666667
263,601,E.T. the Extra-Terrestrial,10500000.0,797307407.0,786807407.0,7493.403876


**Observation:**

- Dragon Ball Super: Broly achieved the highest ROI of approximately 12,400.28%.
- Snow White and the Seven Dwarfs ranked second with an ROI of approximately 12,324.26%.
- The Rocky Horror Picture Show ranked third with an ROI of approximately 12,127.24%.
- Rocky and Gone with the Wind also achieved ROIs above 9,000%.
- All selected movies had production budgets of at least $1 million.

**Conclusion:**

Dragon Ball Super: Broly recorded the highest return on investment among the selected movies. The results demonstrate that movies with relatively small production budgets can generate substantial revenue compared with their initial production costs.

Note: ROI is calculated using production budget and recorded revenue only. It does not include marketing, distribution, or other expenses.

### Problem 5: Analyze the Number of Movies Released Per Year

**Objective:**

Calculate the number of movies released in each year and identify the year with the highest number of movie releases.

**Approach:**
- Extract the year from the `release_date` column.
- Group movies by release year.
- Count the number of movies released in each year.
- Sort the results to identify the year with the highest movie output.

In [10]:
# Convert release_date into datetime format
movies_df["release_date"] = pd.to_datetime(
    movies_df["release_date"],
    errors="coerce"
)

# Extract release year
movies_df["release_year"] = movies_df["release_date"].dt.year

# Count movies released per year
movies_per_year = (
    movies_df.groupby("release_year")["movie_id"]
    .nunique()
    .reset_index(name="movie_count")
    .sort_values(by="movie_count", ascending=False)
)

# Display the results
movies_per_year

,release_year,movie_count
78,2016,121
75,2013,110
79,2017,109
76,2014,105
80,2018,102
...,...,...
5,1938,1
4,1936,1
1,1922,1
2,1927,1


**Observation:**

- 2016 recorded the highest number of movie releases, with 121 movies.
- 2013 ranked second with 110 movies.
- 2017 ranked third with 109 movies.
- 2014 and 2018 recorded 105 and 102 movie releases, respectively.
- The dataset includes movies dating back to 1921, with some early years containing only one movie.

**Conclusion:**

The analysis identifies 2016 as the year with the highest movie output in the TMDB dataset, containing 121 unique movie releases. The results show that the dataset has a stronger concentration of movie releases in more recent years.

### Problem 6: Identify Movies with Runtime Outliers

**Objective:**

Identify movies whose runtimes are statistical outliers compared to the rest of the dataset.

**Approach:**
- Calculate the first quartile (Q1) and third quartile (Q3) of movie runtimes.
- Calculate the Interquartile Range (IQR).
- Determine the lower and upper outlier boundaries.
- Filter movies with runtimes outside these boundaries.



In [12]:
# Remove missing and invalid runtime values
valid_runtime = movies_df[
    movies_df["runtime"].notna() &
    (movies_df["runtime"] > 0)
].copy()

# Calculate Q1 and Q3
Q1 = valid_runtime["runtime"].quantile(0.25)
Q3 = valid_runtime["runtime"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate outlier boundaries
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

# Identify runtime outliers
runtime_outliers = valid_runtime[
    (valid_runtime["runtime"] < lower_bound) |
    (valid_runtime["runtime"] > upper_bound)
]

# Display results
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Total Runtime Outliers:", len(runtime_outliers))

runtime_outliers[
    ["movie_id", "title", "runtime"]
].sort_values(by="runtime", ascending=False)

Q1: 99.0
Q3: 126.0
IQR: 27.0
Lower Bound: 58.5
Upper Bound: 166.5
Total Runtime Outliers: 60


,movie_id,title,runtime
324,791373,Zack Snyder's Justice League,242
1227,770,Gone with the Wind,238
829,311,Once Upon a Time in America,229
1610,947,Lawrence of Arabia,228
1774,665,Ben-Hur,212
587,398978,The Irishman,209
1277,346,Seven Samurai,207
1304,466420,Killers of the Flower Moon,206
187,240,The Godfather Part II,202
20,122,The Lord of the Rings: The Return of the King,201


**Observation:**

- First Quartile (Q1): 99 minutes.
- Third Quartile (Q3): 126 minutes.
- Interquartile Range (IQR): 27 minutes.
- Lower Outlier Bound: 58.5 minutes.
- Upper Outlier Bound: 166.5 minutes.
- A total of 60 movies were identified as runtime outliers.
- Zack Snyder's Justice League has the longest runtime at 242 minutes.
- Gone with the Wind follows with 238 minutes.
- The Garden of Words (46 minutes) and The Guardians of the Galaxy Holiday Special (45 minutes) are unusually short compared to the rest of the dataset.

**Conclusion:**

The IQR method identified 60 movies with unusually short or long runtimes. Most outliers are movies with runtimes exceeding 166.5 minutes. These results highlight unusual runtime patterns in the TMDB dataset, although statistical outliers are not necessarily incorrect records.

### Problem 7: Analyze Primary Genres and Average Popularity

**Objective:**

Identify the primary genre of each movie based on the first genre listed and compare average movie popularity across primary genres.

**Approach:**
- Identify the first genre associated with each movie.
- Merge the primary genre information with the movies dataset.
- Group movies by their primary genre.
- Calculate average popularity for each primary genre.
- Sort the genres by average popularity in descending order.

In [13]:
# Identify the first genre listed for each movie
primary_genres = (
    movie_genres_df
    .drop_duplicates(subset="movie_id", keep="first")
)

# Merge with genre names
primary_genres = primary_genres.merge(
    genres_df,
    on="genre_id",
    how="left"
)

# Merge primary genres with movies
movie_primary_genres = movies_df.merge(
    primary_genres[["movie_id", "genre_name"]],
    on="movie_id",
    how="inner"
)

# Calculate average popularity by primary genre
genre_popularity = (
    movie_primary_genres
    .groupby("genre_name")["popularity"]
    .mean()
    .reset_index(name="average_popularity")
    .sort_values(by="average_popularity", ascending=False)
)

# Display results
genre_popularity

,genre_name,average_popularity
14,Science Fiction,21.261836
11,Music,18.324340
2,Animation,16.811212
7,Family,16.710635
0,Action,16.385188
16,War,15.470317
10,Horror,14.851835
1,Adventure,14.785510
8,Fantasy,13.022920
12,Mystery,12.427131


**Observation:**

- Science Fiction has the highest average popularity at approximately 21.26.
- Music ranks second with an average popularity of 18.32.
- Animation ranks third with 16.81.
- Family and Action follow with average popularity scores of 16.71 and 16.39, respectively.
- Documentary has the lowest average popularity at approximately 7.73.
- Romance and Comedy have similar average popularity scores of approximately 9.98.

**Conclusion:**

Science Fiction is the most popular primary genre based on average TMDB popularity scores in this dataset. Music and Animation also demonstrate relatively high average popularity, while Documentary has the lowest average popularity.

The analysis uses the first listed genre for each movie as its primary genre, so movies are assigned to only one genre for this comparison.

### Problem 8: Analyze the Top 10 Most Prolific Actors

**Objective:**

Identify the top 10 actors based on the number of movies they have appeared in and calculate the average rating of those movies.

**Approach:**
- Group the cast dataset by actor ID and name.
- Count the distinct movies associated with each actor.
- Merge the actor information with the movies dataset.
- Calculate the average movie rating for each actor.
- Sort actors by movie count and retrieve the top 10.

In [14]:
# Count unique movies for each actor
actor_movie_counts = (
    cast_df.groupby(["person_id", "actor_name"])["movie_id"]
    .nunique()
    .reset_index(name="movie_count")
)

# Merge cast data with movie ratings
actor_ratings = cast_df.merge(
    movies_df[["movie_id", "vote_average"]],
    on="movie_id",
    how="inner"
)

# Calculate average rating for each actor
actor_average_ratings = (
    actor_ratings.groupby(["person_id", "actor_name"])["vote_average"]
    .mean()
    .reset_index(name="average_movie_rating")
)

# Combine movie counts and average ratings
actor_analysis = actor_movie_counts.merge(
    actor_average_ratings,
    on=["person_id", "actor_name"],
    how="inner"
)

# Retrieve top 10 prolific actors
top_10_prolific_actors = (
    actor_analysis.sort_values(
        by="movie_count",
        ascending=False
    ).head(10)
)

top_10_prolific_actors

,person_id,actor_name,movie_count,average_movie_rating
679,2231,Samuel L. Jackson,39,6.938692
127,380,Robert De Niro,38,7.228316
106,287,Brad Pitt,38,7.257289
37,85,Johnny Depp,37,6.856973
13,31,Tom Hanks,33,7.289758
413,1245,Scarlett Johansson,32,7.056344
1272,5293,Willem Dafoe,32,7.114781
2674,13240,Mark Wahlberg,32,6.630063
64,192,Morgan Freeman,31,6.963742
161,500,Tom Cruise,31,7.040613


**Observation:**

- Samuel L. Jackson is the most prolific actor, appearing in 39 movies.
- Robert De Niro and Brad Pitt follow with 38 movies each.
- Tom Hanks has the highest average movie rating among the top 10 actors, at approximately 7.29.
- Brad Pitt has an average movie rating of approximately 7.26.
- Mark Wahlberg has the lowest average movie rating among the top 10 actors, at approximately 6.63.

**Conclusion:**

Samuel L. Jackson has the highest number of movie appearances in the dataset. However, Tom Hanks has the highest average movie rating among the top 10 most prolific actors. This analysis compares actors based on both their movie appearances and the average ratings of the movies they appeared in.

### Problem 9: Identify Movie Titles Appearing More Than Once

**Objective:**

Identify movie titles that appear multiple times in the dataset and display their respective release years to investigate possible duplicates or remakes.

**Approach:**
- Group movies by title.
- Identify titles appearing more than once.
- Filter the original movie records using those titles.
- Display the movie titles along with their release years.

In [15]:
# Identify movie titles appearing more than once
duplicate_titles = movies_df[
    movies_df.duplicated(subset="title", keep=False)
].copy()

# Extract release year
duplicate_titles["release_year"] = pd.to_datetime(
    duplicate_titles["release_date"],
    errors="coerce"
).dt.year

# Display duplicate titles and release years
duplicate_titles[
    ["movie_id", "title", "release_year"]
].sort_values(by="title")

,movie_id,title,release_year
878,377,A Nightmare on Elm Street,1984
1928,23437,A Nightmare on Elm Street,2010
261,812,Aladdin,1992
339,420817,Aladdin,2019
166,12155,Alice in Wonderland,2010
...,...,...,...
1737,60935,The Thing,2011
744,861,Total Recall,1990
860,64635,Total Recall,2012
252,1858,Transformers,2007


**Observation:**

- A total of 100 movie records were identified with titles appearing more than once.
- A Nightmare on Elm Street appears in 1984 and 2010.
- Aladdin appears in 1992 and 2019.
- Total Recall appears in 1990 and 2012.
- Transformers appears in 1987 and 2007.
- These repeated titles may represent remakes, reboots, or different movies sharing the same title.

**Conclusion:**

The analysis identifies movie titles that occur multiple times in the dataset. Comparing their movie IDs and release years helps distinguish possible remakes and reboots from potential duplicate records. Repeated titles alone are not sufficient evidence of duplicate movies.

### Problem 10: Identify Unusual Popularity and Vote Count Relationships

**Objective:**

Identify movies with unusual combinations of popularity scores and vote counts, such as high popularity with low vote counts or low popularity with high vote counts.

**Approach:**
- Select valid popularity and vote count values.
- Calculate Z-scores for popularity and vote count.
- Identify movies with unusually high popularity but low vote counts, or unusually low popularity but high vote counts.
- Display the identified movies and their corresponding values.

In [20]:
# Calculate Z-scores for popularity and vote count

popularity_analysis["popularity_zscore"] = (
    (popularity_analysis["popularity"] -
     popularity_analysis["popularity"].mean()) /
    popularity_analysis["popularity"].std()
)

popularity_analysis["vote_count_zscore"] = (
    (popularity_analysis["vote_count"] -
     popularity_analysis["vote_count"].mean()) /
    popularity_analysis["vote_count"].std()
)

# Identify unusual combinations
unusual_movies = popularity_analysis[
    (
        (popularity_analysis["popularity_zscore"] > 2) &
        (popularity_analysis["vote_count_zscore"] < 0)
    ) |
    (
        (popularity_analysis["popularity_zscore"] < 0) &
        (popularity_analysis["vote_count_zscore"] > 2)
    )
].copy()

# Display results
unusual_movies[
    [
        "movie_id",
        "title",
        "popularity",
        "vote_count",
        "popularity_zscore",
        "vote_count_zscore"
    ]
].sort_values(
    by="popularity_zscore",
    ascending=False
)

,movie_id,title,popularity,vote_count,popularity_zscore,vote_count_zscore
2388,1275779,Disclosure Day,394.3346,2328,21.019194,-0.779558
1315,1339713,Obsession,364.8722,4203,19.392273,-0.401458
2248,1083381,Backrooms,258.4743,2483,13.516953,-0.748302
1465,936075,Michael,108.7248,3846,5.247747,-0.473449
2452,931285,Mortal Kombat II,99.5231,2260,4.739627,-0.793271
1707,1226863,The Super Mario Galaxy Movie,97.1210,3343,4.606982,-0.574880
1378,83533,Avatar: Fire and Ash,95.5319,4032,4.519232,-0.435941
2295,1304313,Lee Cronin's The Mummy,79.4601,2430,3.631743,-0.758990
1776,1084242,Zootopia 2,63.1069,3195,2.728715,-0.604725
1754,822119,Captain America: Brave New World,60.3616,3233,2.577119,-0.597062


In [21]:
popularity_analysis[
    ["popularity", "vote_count"]
].corr()

,popularity,vote_count
popularity,1.00000,0.36085
vote_count,0.36085,1.00000


### Observations

- The correlation between popularity and vote count is approximately **0.36085**, indicating a positive but relatively weak-to-moderate relationship.
- A total of **18 movies** were identified with unusual combinations of popularity and vote count using Z-score thresholds.
- **12 movies** had exceptionally high popularity scores (above 2 standard deviations) despite having below-average vote counts.
- **6 movies** had below-average popularity scores despite having unusually high vote counts (above 2 standard deviations).
- *Disclosure Day*, *Obsession*, and *Backrooms* were among the movies with exceptionally high popularity and comparatively low vote counts.
- *The Hunger Games*, *It*, and *The Silence of the Lambs* were among the movies with relatively low popularity but very high vote counts.

### Conclusion

Popularity and vote count have a positive relationship, but the relationship is not particularly strong. Some movies receive high popularity scores despite having fewer votes, while others accumulate substantial votes without correspondingly high popularity scores.

This suggests that popularity and vote count measure different aspects of audience engagement and should not be treated as interchangeable metrics.

**Methodology:** Z-scores were used to identify unusual combinations. Movies with popularity Z-scores above 2 and vote count Z-scores below 0, or popularity Z-scores below 0 and vote count Z-scores above 2, were selected for further analysis.